# ML_A6 — Support Vector Machines (SVM)

📝 **Modalidad: Trabajo autónomo — practica a tu ritmo.**

**Versión:** 2025-1 | **Modificado:** 2026-05-03

---

## 🎯 Contexto del Problema

**Dataset:** `wine` de sklearn
- **Tamaño:** 178 muestras de vinos italianos
- **Features:** 13 características químicas (alcohol, ácido málico, ceniza, etc.)
- **Clases:** 3 variedades de uva (0, 1, 2)
- **Tarea:** Clasificación multiclase — identificar la variedad según análisis químico

**Pregunta central:**
> ¿Puede un SVM con kernel RBF superar a un SVM lineal en este problema?
> ¿Qué hiperparámetros lo hacen posible?

**Restricciones:**
- ✅ Normalizar con `StandardScaler` (obligatorio para SVM)
- ✅ Buscar hiperparámetros con `GridSearchCV` (sin data leakage)
- ✅ Comparar al menos 2 kernels y justificar la elección
- ❌ No usar redes neuronales ni métodos de ensemble

---

## Setup (NO MODIFICAR)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, validation_curve
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Carga dataset
wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = pd.Series(wine.target, name='variedad')

# Split estratificado 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

print("✅ Setup completo")
print(f"   Dataset: Wine — {X.shape[0]} muestras, {X.shape[1]} features, 3 clases")
print(f"   Clases: {wine.target_names}")
print(f"   Train: {len(X_train)} | Test: {len(X_test)}")
print(f"   Distribución y_train: {dict(zip(*np.unique(y_train, return_counts=True)))}")

## Sección 1 — EDA y Preprocesamiento

**Objetivo:** Entender las features, escalarlas y preparar el pipeline para SVM.

In [ ]:
# TODO 1: Análisis exploratorio básico
# a) Imprime las estadísticas descriptivas de X_train (X_train.describe())
# b) Visualiza la distribución de clases en y_train (barplot o countplot)
# c) Crea un heatmap de correlación entre features (usa X_train.corr())

# a) Estadísticas
# print(X_train.describe().round(2))

# b) Distribución de clases
# fig, ax = plt.subplots(figsize=(6, 4))
# ...

# c) Heatmap de correlación
# fig, ax = plt.subplots(figsize=(12, 10))
# sns.heatmap(X_train.corr(), annot=False, cmap='coolwarm', center=0, ax=ax)
# ...


In [ ]:
# TODO 2: Normalizar con StandardScaler
# IMPORTANTE: fit SOLO con X_train, transform en X_train y X_test por separado

# scaler = StandardScaler()
# X_train_sc = scaler.fit_transform(X_train)
# X_test_sc  = scaler.transform(X_test)

# print(f"✅ Media X_train_sc: {X_train_sc.mean():.4f} (≈ 0)")
# print(f"   Desv. Est. X_train_sc: {X_train_sc.std():.4f} (≈ 1)")

# Tests de sanidad Sección 1 (NO MODIFICAR)
# assert X_train_sc.shape == (X_train.shape[0], 13), "Shape de X_train_sc incorrecto"
# assert X_test_sc.shape  == (X_test.shape[0],  13), "Shape de X_test_sc incorrecto"
# assert abs(X_train_sc.mean()) < 0.01, "Media no cercana a 0"
# print("\n✅ Sanity checks Sección 1: PASSED")


## Sección 2 — SVM Lineal: Baseline

**Objetivo:** Establece un baseline con kernel lineal antes de probar kernels no-lineales.

In [ ]:
# TODO 3: SVM Lineal (baseline)
# a) Entrena SVC(kernel='linear', C=1.0, random_state=RANDOM_STATE) en X_train_sc
# b) Predice en X_test_sc
# c) Calcula accuracy y F1-macro
# d) Imprime classification_report con target_names=wine.target_names

# svm_lin = SVC(kernel='linear', C=1.0, random_state=RANDOM_STATE)
# svm_lin.fit(X_train_sc, y_train)
# y_pred_lin = svm_lin.predict(X_test_sc)
# acc_lin = accuracy_score(y_test, y_pred_lin)
# f1_lin  = f1_score(y_test, y_pred_lin, average='macro')

# print(f"SVM Lineal — Accuracy: {acc_lin:.4f} | F1-macro: {f1_lin:.4f}")
# print()
# print(classification_report(y_test, y_pred_lin, target_names=wine.target_names))


In [ ]:
# TODO 4 [PhD]: Analiza los vectores de soporte del SVM lineal
# Para cada uno de los 3 subclasificadores binarios (OvR):
# - ¿Cuántos SVs tiene?
# - ¿Qué porcentaje del training es?
# n_sv_lin = svm_lin.n_support_
# print(f"SVs por clase: {n_sv_lin}")
# print(f"Total SVs: {n_sv_lin.sum()} / {len(X_train)} ({100*n_sv_lin.sum()/len(X_train):.1f}%)")


In [ ]:
# Tests de sanidad Sección 2 (NO MODIFICAR)
# assert acc_lin > 0.80, "SVM Lineal accuracy muy bajo"
# assert f1_lin  > 0.80, "SVM Lineal F1 muy bajo"
# print("\n✅ Sanity checks Sección 2: PASSED")


## Sección 3 — Kernel SVM: ¿Puede RBF superar al lineal?

**Objetivo:** Explorar el efecto de los kernels y entender el rol de C y γ.

In [ ]:
# TODO 5: Entrenar SVM RBF con C=1.0 y gamma='scale'
# Compara con el baseline lineal

# svm_rbf = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=RANDOM_STATE)
# svm_rbf.fit(X_train_sc, y_train)
# y_pred_rbf = svm_rbf.predict(X_test_sc)
# acc_rbf = accuracy_score(y_test, y_pred_rbf)
# f1_rbf  = f1_score(y_test, y_pred_rbf, average='macro')

# print(f"SVM RBF   — Accuracy: {acc_rbf:.4f} | F1-macro: {f1_rbf:.4f}")
# print(f"SVM Lineal— Accuracy: {acc_lin:.4f} | F1-macro: {f1_lin:.4f}")
# print(f"Diferencia de accuracy: {acc_rbf - acc_lin:+.4f}")


In [ ]:
# TODO 6: Tabla comparativa con Cross-Validation (cv=5, scoring='f1_macro')
# Para kernels: 'linear', 'rbf', 'poly'
# Crea DataFrame con: kernel, cv_f1_mean, cv_f1_std, test_accuracy

# kernels_cfg = {
#     'linear': SVC(kernel='linear', C=1.0, random_state=RANDOM_STATE),
#     'rbf':    SVC(kernel='rbf', C=1.0, gamma='scale', random_state=RANDOM_STATE),
#     'poly':   SVC(kernel='poly', C=1.0, degree=3, gamma='scale', random_state=RANDOM_STATE),
# }

# rows = []
# for kname, model in kernels_cfg.items():
#     model.fit(X_train_sc, y_train)
#     cv = cross_val_score(model, X_train_sc, y_train, cv=5, scoring='f1_macro')
#     rows.append({'kernel': kname, 'cv_f1_mean': cv.mean(), 'cv_f1_std': cv.std(),
#                  'test_acc': accuracy_score(y_test, model.predict(X_test_sc))})
# df_kernels = pd.DataFrame(rows)
# print(df_kernels.round(4))


In [ ]:
# TODO 7 [PhD]: Curvas de validación para C y gamma (kernel RBF)
# Usa validation_curve para:
# a) Variar C ∈ [0.001, 0.01, 0.1, 1, 10, 100, 1000] con gamma='scale'
# b) Variar gamma ∈ [0.001, 0.01, 0.1, 1, 10, 100] con C=1.0
# Grafica train_score y val_score en función del parámetro

# C_range = np.logspace(-3, 3, 7)
# train_scores_C, val_scores_C = validation_curve(
#     SVC(kernel='rbf', gamma='scale', random_state=RANDOM_STATE),
#     X_train_sc, y_train, param_name='C', param_range=C_range, cv=5, scoring='f1_macro')
# ...


In [ ]:
# Tests de sanidad Sección 3 (NO MODIFICAR)
# assert acc_rbf > 0.80, "SVM RBF accuracy muy bajo"
# assert len(df_kernels) == 3, "Tabla debe tener 3 kernels"
# print("\n✅ Sanity checks Sección 3: PASSED")


## Sección 4 — Grid Search: Encontrando los Mejores Hiperparámetros

**Objetivo:** Optimizar C y γ sin data leakage usando GridSearchCV.

In [ ]:
# TODO 8: Grid Search completo con Pipeline (evita data leakage)
# Usa Pipeline([('scaler', StandardScaler()), ('svm', SVC(...))])
# Busca sobre: C ∈ [0.1, 1, 10, 100] y gamma ∈ ['scale', 0.01, 0.1]
# kernel='rbf', cv=5, scoring='f1_macro'

# pipeline_gs = Pipeline([
#     ('scaler', StandardScaler()),
#     ('svm', SVC(kernel='rbf', random_state=RANDOM_STATE))
# ])
# param_grid = {
#     'svm__C':     [0.1, 1, 10, 100],
#     'svm__gamma': ['scale', 0.01, 0.1],
# }
# gs = GridSearchCV(pipeline_gs, param_grid, cv=5, scoring='f1_macro', n_jobs=-1)
# gs.fit(X_train, y_train)  # OJO: X_train sin escalar (el pipeline lo hace internamente)

# print(f"Mejores parámetros: {gs.best_params_}")
# print(f"Mejor CV F1-macro:  {gs.best_score_:.4f}")

# best_pipeline = gs.best_estimator_
# y_pred_best = best_pipeline.predict(X_test)
# print(f"Test Accuracy:      {accuracy_score(y_test, y_pred_best):.4f}")
# print(f"Test F1-macro:      {f1_score(y_test, y_pred_best, average='macro'):.4f}")


In [ ]:
# TODO 9: Heatmap de resultados del Grid Search
# Visualiza mean CV F1-macro para cada combinación de C y gamma

# cv_df = pd.DataFrame(gs.cv_results_)
# pivot = cv_df.pivot_table(
#     index='param_svm__C', columns='param_svm__gamma', values='mean_test_score')
# fig, ax = plt.subplots(figsize=(7, 5))
# sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlGn', ax=ax)
# ax.set_title('Grid Search — CV F1-macro (Pipeline)')
# ...


In [ ]:
# Tests de sanidad Sección 4 (NO MODIFICAR)
# assert accuracy_score(y_test, y_pred_best) > 0.90, "Accuracy del mejor SVM < 0.90"
# assert f1_score(y_test, y_pred_best, average='macro') > 0.90, "F1 del mejor SVM < 0.90"
# print("\n✅ Sanity checks Sección 4: PASSED")


## Sección 5 — Análisis Final del Mejor Modelo

**Objetivo:** Interpretar el clasificador entrenado y comunicar los resultados.

In [ ]:
# TODO 10: Reporte y confusion matrix del mejor modelo
# a) Imprime classification_report del mejor pipeline
# b) Visualiza la confusion matrix
# c) Compara con el baseline lineal (Sección 2) en una tabla

# print("CLASSIFICATION REPORT — Mejor SVM (Grid Search):")
# print(classification_report(y_test, y_pred_best, target_names=wine.target_names))

# Confusion matrix
# fig, ax = plt.subplots(figsize=(6, 5))
# ConfusionMatrixDisplay(
#     confusion_matrix(y_test, y_pred_best),
#     display_labels=wine.target_names
# ).plot(ax=ax, colorbar=False, cmap='Blues')
# ax.set_title('Confusion Matrix — Mejor SVM', fontweight='bold')
# plt.tight_layout()
# plt.show()

# Tabla comparativa
# comparison = pd.DataFrame({
#     'Modelo': ['SVM Lineal (C=1)', 'SVM RBF (optimizado)'],
#     'Test Accuracy': [acc_lin, accuracy_score(y_test, y_pred_best)],
#     'Test F1-macro': [f1_lin,  f1_score(y_test, y_pred_best, average='macro')]
# })
# print(comparison.round(4))


In [ ]:
# TODO 11: Visualización de fronteras de decisión en 2D (PCA)
# a) Aplica PCA(n_components=2) a X_train_sc y X_test_sc
# b) Entrena un SVM (con los mejores parámetros) en el espacio PCA
# c) Visualiza la frontera de decisión 2D con colores por clase

# pca = PCA(n_components=2, random_state=RANDOM_STATE)
# X_train_2d = pca.fit_transform(X_train_sc)
# X_test_2d  = pca.transform(X_test_sc)

# svm_2d = SVC(kernel='rbf', C=10, gamma='scale', random_state=RANDOM_STATE)
# svm_2d.fit(X_train_2d, y_train)

# h = 0.05
# x_min, x_max = X_train_2d[:,0].min()-0.5, X_train_2d[:,0].max()+0.5
# y_min, y_max = X_train_2d[:,1].min()-0.5, X_train_2d[:,1].max()+0.5
# xx, yy = np.meshgrid(np.arange(x_min,x_max,h), np.arange(y_min,y_max,h))
# Z = svm_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
# fig, ax = plt.subplots(figsize=(9, 6))
# ax.contourf(xx, yy, Z, cmap='RdYlBu', alpha=0.3)
# scatter = ax.scatter(X_test_2d[:,0], X_test_2d[:,1], c=y_test, cmap='RdYlBu',
#                      edgecolors='k', s=70, zorder=3)
# ax.scatter(svm_2d.support_vectors_[:,0], svm_2d.support_vectors_[:,1],
#            s=200, facecolors='none', edgecolors='gold', linewidths=2, zorder=4,
#            label='Vectores de soporte')
# ax.set_title(f"SVM RBF — Frontera 2D (PCA)\n"
#              f"Acc(2D)={accuracy_score(y_test, svm_2d.predict(X_test_2d)):.2f}")
# ax.legend(); plt.tight_layout(); plt.show()


## Sección 6 — Autoevaluación

Antes de terminar, verifica tu comprensión respondiendo estas preguntas.

### Checklist General (Todos)
- [ ] Cargué el dataset `wine` correctamente
- [ ] Normalicé con StandardScaler (fit en train, transform en test)
- [ ] Entrené y comparé al menos dos kernels (lineal y RBF)
- [ ] Usé GridSearchCV con Pipeline para evitar data leakage
- [ ] Interpreté los vectores de soporte y su porcentaje del training

### Checklist 🔵 Pregrado
- [ ] Comprendo por qué normalizar es obligatorio en SVM
- [ ] Sé explicar la diferencia entre hard margin y soft margin
- [ ] Entiendo el rol de C (regularización) y γ (ancho del kernel)
- [ ] Puedo interpretar una confusion matrix multiclase
- [ ] Sé cuándo elegir SVM sobre otros clasificadores

### Checklist 🟡 Doctorado
- [ ] Derivé la formulación dual del SVM (en papel o en markdown)
- [ ] Interpreté las curvas de validación para C y γ
- [ ] Analicé el número de SVs por clase en OvR multiclase
- [ ] Expliqué la relación entre γ, VC-dimension y generalización
- [ ] Comprendí por qué el Pipeline evita data leakage en CV

### Reflexión Final (máximo 150 palabras)

1. ¿El kernel RBF superó al lineal? ¿Qué indica esto sobre la naturaleza del problema?
2. ¿Cuál fue el mayor desafío en este assignment?
3. ¿Qué concepto quedó menos claro?

*[Espacio para tu respuesta]*

## Tests de Sanidad Globales (NO MODIFICAR)

In [ ]:
# Tests globales — ejecutar solo si completaste todas las secciones
# print("="*65)
# print("SANITY CHECKS GLOBALES")
# print("="*65)

# assert X_train_sc.shape == (X_train.shape[0], 13), "Shape X_train_sc"
# assert X_test_sc.shape  == (X_test.shape[0],  13), "Shape X_test_sc"
# assert abs(X_train_sc.mean()) < 0.01, "Media X_train_sc"
# assert acc_lin > 0.80, "SVM Lineal accuracy"
# assert acc_rbf > 0.80, "SVM RBF accuracy"
# assert accuracy_score(y_test, y_pred_best) > 0.90, "Mejor SVM accuracy"
# assert f1_score(y_test, y_pred_best, average='macro') > 0.90, "Mejor SVM F1"
# print("✅ Todos los sanity checks pasaron")

# print()
# print("Resumen Final:")
# print(f"  SVM Lineal:       Acc={acc_lin:.4f}, F1={f1_lin:.4f}")
# print(f"  SVM RBF (C=1):    Acc={acc_rbf:.4f}, F1={f1_rbf:.4f}")
# print(f"  SVM RBF (óptimo): Acc={accuracy_score(y_test, y_pred_best):.4f}, "
#       f"F1={f1_score(y_test, y_pred_best, average='macro'):.4f}")
# print(f"  Mejores params: {gs.best_params_}")


---
## Nota para Prueba Escrita

Este notebook es **trabajo autónomo formativo**. La calificación final viene de una **prueba escrita en papel**.

**Conceptos clave que debes dominar:**
1. **Margen máximo:** ¿Por qué maximizarlo mejora la generalización?
2. **Parámetro C:** trade-off entre margen y violaciones — ¿cuándo C grande vs C pequeño?
3. **Kernel trick:** proyección implícita — ¿qué garantiza el teorema de Mercer?
4. **Gamma:** ¿cómo afecta la complejidad del clasificador?
5. **Data leakage:** ¿por qué el scaler debe estar dentro del pipeline en GridSearchCV?

**Errores comunes a evitar:**
- ❌ No normalizar antes de SVM
- ❌ Hacer fit del scaler con X_test (data leakage)
- ❌ No estratificar el split en datasets con clases desbalanceadas
- ❌ Elegir hiperparámetros con test set directamente (usar CV)

---